In [69]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler,StandardScaler,OneHotEncoder
from category_encoders.count import CountEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
data_path = Path("C:\\Users\\Jay Kanakia\\.cache\\kagglehub\\datasets\\undefinenull\\million-song-dataset-spotify-lastfm\\versions\\1")

song_data_path = data_path/"Music Info.csv"
user_data_path = data_path/"User Listening History.csv"

In [3]:
# load the song data

df_songs = pd.read_csv(song_data_path)

df_songs.sample(3)

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
31808,TRKAMDY128F42559B3,This Is Nowhere,The Airborne Toxic Event,https://p.scdn.co/mp3-preview/9cb2c339fb502b6f...,2nh4EsLdchvBdY8Y4DYOHR,"indie, indie_rock, post_punk",NaN,2009,169746,0.329,...,2,-2.769,1,0.0743,0.073000,0.000000,0.305,0.507,159.859,4
44413,TRRGXHG128F4266392,Oh Atlanta,Little Feat,https://p.scdn.co/mp3-preview/f9ea50675435df83...,0KT50Gp03rD633HtU8A8Sp,"rock, blues_rock",Country,2004,292640,0.268,...,5,-5.724,1,0.0607,0.564000,0.002160,0.774,0.550,164.317,4
4546,TRAOJMT128F92FD7E9,Tristeza Maleza,Manu Chao,https://p.scdn.co/mp3-preview/e0c359be38a19102...,0D5kDzncleh3tjh877IbFm,"reggae, ska",Latin,2007,174360,0.623,...,4,-6.969,0,0.0278,0.000205,0.000787,0.186,0.898,150.066,4


In [4]:
# duplicates in dataset

df_songs.duplicated().sum()

np.int64(0)

In [5]:
# duplicates based on spotify_id

df_songs.duplicated(subset=["spotify_id","year","duration_ms"]).sum()

np.int64(9)

In [6]:
# drop duplicated rows

df_songs.drop_duplicates(subset=["spotify_id","year","duration_ms"],inplace=True)

df_songs.duplicated(subset=["spotify_id","year","duration_ms"]).sum()

np.int64(0)

In [7]:
df_songs.reset_index(drop=True,inplace=True)

In [8]:
# remove columns not required for content filtering

columns_to_remove = ["track_id","name","spotify_preview_url","spotify_id","genre"]

df_content_filtering = df_songs.drop(columns=columns_to_remove)

df_content_filtering.sample(3)

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
22315,The Birthday Massacre,"electronic, female_vocalists, industrial, gothic",2007,242760,0.527,0.927,2,-4.914,1,0.0392,0.000033,0.902,0.664,0.369,100.014,4
239,The Smiths,"rock, alternative, indie, pop, alternative_roc...",2008,243573,0.522,0.734,4,-6.899,1,0.0269,0.042500,0.000,0.172,0.864,135.991,4
12084,Late of the Pier,"electronic, alternative, indie, experimental, ...",2008,77866,0.456,0.631,4,-5.789,0,0.0416,0.003100,0.514,0.357,0.406,114.771,3


In [9]:
# check for missing values

df_content_filtering.isnull().sum()

artist                 0
tags                1126
year                   0
duration_ms            0
danceability           0
energy                 0
key                    0
loudness               0
mode                   0
speechiness            0
acousticness           0
instrumentalness       0
liveness               0
valence                0
tempo                  0
time_signature         0
dtype: int64

In [10]:
# fill the tags column missing values with string "no_tags"

df_content_filtering.fillna({'tags':'no_tags'},inplace=True)

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,Nirvana,"rock, alternative, alternative_rock, 90s, grunge",1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,Franz Ferdinand,"rock, alternative, indie, alternative_rock, in...",2004,237026,0.279,0.664,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,Radiohead,"rock, alternative, indie, alternative_rock, in...",2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50669,アンティック-珈琲店-,no_tags,2008,273440,0.438,0.933,6,-3.062,0,0.1650,0.003120,0.000000,0.1300,0.421,166.956,4
50670,ACIDMAN,"rock, alternative_rock, japanese, cover",2004,275133,0.351,0.693,0,-6.811,1,0.1200,0.000940,0.000049,0.1920,0.450,200.350,4
50671,coldrain,"metal, metalcore, post_hardcore",2014,254826,0.434,0.975,10,-3.092,0,0.2680,0.000108,0.001410,0.1630,0.282,158.025,4
50672,アンティック-珈琲店-,no_tags,2008,243293,0.513,0.902,4,-3.914,0,0.0530,0.000715,0.001350,0.0571,0.618,109.923,4


In [11]:
df_content_filtering.isnull().sum()

artist              0
tags                0
year                0
duration_ms         0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
dtype: int64

In [12]:
# number of unique artist

df_content_filtering['artist'].str.lower().nunique()

8317

In [13]:
# number of unique year values

df_content_filtering.loc[:,'year'].nunique()

75

In [112]:
# min and max values of numerical columns

df_content_filtering.select_dtypes(include='number').describe().loc[['min','max']]

,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
min,1900.0,1439.0,0.000,0.0,0.0,-60.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.0
max,2022.0,3816373.0,0.986,1.0,11.0,3.642,1.0,0.954,0.996,0.999,0.999,0.993,238.895,5.0


In [15]:
(
    df_content_filtering
    .loc[:,'tags']
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
)

tags
rock            10681
indie            7284
electronic       6592
alternative      6271
pop              4650
                ...  
dark_ambient      602
japanese          489
polish            411
j_pop             213
russian           127
Name: count, Length: 101, dtype: int64

In [16]:
(
    df_content_filtering
    .loc[:,'tags']
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .loc[lambda x : x>1000]
)

tags
rock            10681
indie            7284
electronic       6592
alternative      6271
pop              4650
                ...  
ska              1088
gothic_metal     1072
grindcore        1040
french           1018
nu_metal         1006
Name: count, Length: 86, dtype: int64

In [31]:
# columns to transform

frequency_encode_cols = ['year']
ohe_cols = ['artist','time_signature','key']
tfidf_col = ['tags']
standard_scaler_cols = ['duration_ms','loudness','tempo']
min_max_scaler_cols = ['danceability','energy','speechiness','acousticness','instrumentalness','liveness','valence']


In [55]:
# transform the data

transformer = ColumnTransformer(
    transformers=[
        ('frequency_encode',CountEncoder(normalize=True,return_df=True),frequency_encode_cols),
        ('ohe',OneHotEncoder(handle_unknown='ignore'),ohe_cols),
        ('tfidf',TfidfVectorizer(max_features=85),tfidf_col[0]),
        ('standard_scale',StandardScaler(),standard_scaler_cols),
        ('min_max_scale',MinMaxScaler(),min_max_scaler_cols)
    ],
    remainder='passthrough',n_jobs=-1,verbose_feature_names_out=False
)

transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [56]:
# fit the transformer

transformer.fit(df_content_filtering)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [58]:
transformed_df = transformer.transform(df_content_filtering)

In [59]:
transformed_df.shape

(50674, 8431)

In [60]:
transformed_df

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 907911 stored elements and shape (50674, 8431)>

In [62]:
df_content_filtering.shape

(50674, 16)

In [66]:
# build input vector

song_input = df_content_filtering[df_songs['name'] == "Whenever, Wherever"]

song_input

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,Shakira,"rock, pop, female_vocalists, singer_songwriter...",2012,196826,0.787,0.828,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [67]:
# input vector to calculate similarity

input_vector = transformer.transform(song_input)

input_vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20 stored elements and shape (1, 8431)>

In [68]:
input_vector.shape

(1, 8431)

In [70]:
similarity_score = cosine_similarity(transformed_df,input_vector)

similarity_score

array([[0.99999914],
       [0.99999847],
       [0.99999921],
       ...,
       [0.99999877],
       [0.9999992 ],
       [0.99999891]], shape=(50674, 1))

In [71]:
similarity_score.shape

(50674, 1)

In [87]:
top_10_songs_indexes = np.argsort(similarity_score,axis=0).ravel()[-11:][::-1]

top_10_songs_indexes

array([ 1025, 12305,  6046,  6129, 17241,  6133,  7172,  6121,  6526,
       38383,  6287])

In [89]:
top_10_songs_names = df_songs.iloc[top_10_songs_indexes]

top_10_songs_names

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.29800,0.000005,0.2060,0.860,107.674,4
12305,TRYFVKK128F4240FE8,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...,0HiJFRxWme9myvUiDlqQ8q,"pop, experimental, singer_songwriter, dance",NaN,2001,221240,0.887,...,1,-5.535,0,0.0431,0.14400,0.000590,0.1230,0.399,129.943,4
6046,TRAAKDG128F42A0ECB,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,01Yj2MCGpjZs34PRlGgz4K,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,217453,0.777,...,10,-5.867,0,0.0734,0.28400,0.000000,0.4300,0.760,100.003,4
6129,TRBAUVN128F932FEF8,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...,095uakqDYR50Uza0mxvPWB,"pop, female_vocalists, dance, 00s",Pop,2014,211786,0.751,...,1,-5.351,0,0.0435,0.34000,0.000018,0.2550,0.886,95.045,4
17241,TRMBDIR128F4279C1F,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...,1BhxPx4evrx8X02RHGrLdi,"pop, dance, rnb, 00s",Rock,2007,182680,0.718,...,1,-3.959,0,0.0360,0.35300,0.000390,0.1020,0.805,117.067,4
6133,TRDXCSH128F92ED4A1,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...,09mkdGhqb5ySKVsnkx9hy2,"pop, female_vocalists, dance, soul, hip_hop, r...",NaN,2001,207733,0.835,...,1,-4.364,0,0.2840,0.00247,0.000000,0.1710,0.586,103.358,4
7172,TRLQBEN128F92E7415,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...,0RgHkNpqtAHGGBVro6mmqD,"pop, female_vocalists",Country,2016,188493,0.741,...,1,-5.362,0,0.1080,0.01950,0.000000,0.0822,0.735,107.960,4
6121,TRGZIMZ128F930A016,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...,0rpndqrkU9y9nckNCfjcq6,"pop, female_vocalists, dance, 80s",NaN,2009,242946,0.708,...,1,-4.736,0,0.0362,0.39200,0.000001,0.0561,0.968,99.953,4
6526,TRXHWIA128E078A9BB,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...,0BmUCeyXpTrUVahHKVFxuB,"pop, female_vocalists, dance, 80s, new_wave",NaN,1984,209573,0.665,...,1,-5.828,0,0.0278,0.25000,0.008550,0.0537,0.932,108.429,4
38383,TRWUWRZ128F42ADA4A,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...,2ObxMmMaDINr0ynkqW2BlY,"pop, female_vocalists, guitar, pop_rock",Pop,2005,242760,0.689,...,1,-7.427,1,0.0286,0.18000,0.000038,0.0844,0.548,96.098,4


In [102]:
def recommend(song_name, songs_data, transformed_data, k=10):
    
    """
    Recommends top k songs similar to the given song based on content-based filtering.

    Parameters:
    song_name (str): The name of the song to base the recommendations on.
    songs_data (DataFrame): The DataFrame containing song information.
    transformed_data (ndarray): The transformed data matrix for similarity calculations.
    k (int, optional): The number of similar songs to recommend. Default is 10.

    Returns:
    DataFrame: A DataFrame containing the top k recommended songs with their names, artists, and Spotify preview URLs.
    """

    # filter out the song from data
    song_row = songs_data[songs_data['name'] == song_name]

    if song_row.empty:
        print('Song not found in the dataset')
    else:
        # get the index of the song
        song_index = song_row.index[0]
        print(f'Index of the song entered is {song_index}')

        # generate the input vector
        input_vector = transformed_data[song_index]

        # calculate similarity score
        similarity_score = cosine_similarity(input_vector,transformed_data)
        print(f'Similarity score shape is : {similarity_score.shape}')

        # get the top k songs
        top_k_songs_indexes = np.argsort(similarity_score,axis=1).ravel()[-k-1:][::-1]
        print(f"Top {k} songs indexes are {top_k_songs_indexes}")

        # get the top k songs names
        top_k_songs_names = songs_data.iloc[top_k_songs_indexes]

        # print the top k songs
        top_k_list = top_10_songs_names[['name','artist','spotify_preview_url']].reset_index(drop=True)

        return top_k_list

In [103]:
recommend("Whenever, Wherever",songs_data=df_songs,transformed_data=transformed_df,k=10)

Index of the song entered is 1025
Similarity score shape is : (1, 50674)
Top 10 songs indexes are [ 1025 12305  6046  6129 17241  6133  7172  6121  6526 38383  6287]


,name,artist,spotify_preview_url
0,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...
1,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...
2,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...
3,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...
4,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...
5,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...
6,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...
7,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...
8,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...
9,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...
